---
# `Contextual Compressor Retriever`
---

Eg> What is Photosynthesis?
- Retrieval document: The grand canyon is a famours nature site, photosynthesis is how plants converts light into energy. Tourist visit every year
- Use RecursiveCharacterTextSplitter
- Trim the part or compression
- This Retriever is an Advanced retriever that Improves Retrieval Quality by compressing document after retrieval. As keeping only the Relevant content based on the user's query.


How it works?
- User's query --> Retriever --> Fetch documents from Vector store --> D1, D2 etc --> LLM --> Compression into D1' , D2' etc --> 

# `Detailed Notes`

# Contextual Compression Retriever

A **Contextual Compression Retriever** is a LangChain retrieval technique that first retrieves documents and then **compresses/filters those documents to keep only the parts relevant to the user's query**.

The core idea is:

> **Retrieve broadly, then remove irrelevant information before sending the context to the LLM.**

---

## 1. Why Do We Need Contextual Compression?

Suppose a user asks:

> **"What is the learning rate of the model?"**

Your retriever finds a document of 2,000 words:

```text
Document
├── Model architecture
├── Dataset
├── Training process
├── Learning rate       ← Relevant
├── Optimizer
├── Evaluation
├── Deployment
└── Future work
```

A normal retriever may pass the **entire document** to the LLM.

That's inefficient.

With contextual compression:

```text
User Query
    ↓
Retrieve Document
    ↓
Compression
    ↓
Keep only relevant information
    ↓
LLM
```

The resulting context might contain only:

```text
"The model uses a learning rate of 0.001."
```

---

# 2. Normal Retriever vs Contextual Compression

### Normal Retriever

```text
Query
  ↓
Retriever
  ↓
Relevant Documents
  ↓
LLM
```

### Contextual Compression Retriever

```text
Query
  ↓
Base Retriever
  ↓
Retrieved Documents
  ↓
Compressor
  ↓
Relevant Content
  ↓
LLM
```

The important addition is:

```text
Retriever → Compressor
```

---

# 3. Why Is It Called "Contextual Compression"?

There are two important words.

### Contextual

The compression is performed **with respect to the user's query**.

For example:

```text
Document:
"Python was created by Guido van Rossum.
Python is dynamically typed.
Python is widely used in AI.
Python was first released in 1991."
```

Query:

```text
"When was Python first released?"
```

The compressor should focus on:

```text
"Python was first released in 1991."
```

The query provides the **context** for deciding what is relevant.

---

### Compression

The original content:

```text
1000 words
```

might become:

```text
100 words
```

while preserving the information needed to answer the question.

---

# 4. Architecture

```text
                       User Query
                           │
                           ▼
                  ┌─────────────────┐
                  │ Base Retriever  │
                  └────────┬────────┘
                           │
                           ▼
                    Retrieved Docs
                           │
                           ▼
                  ┌─────────────────┐
                  │   Compressor    │
                  └────────┬────────┘
                           │
                           ▼
                  Relevant Content
                           │
                           ▼
                          LLM
                           │
                           ▼
                         Answer
```

So the Contextual Compression Retriever is essentially:

```text
Base Retriever + Compressor
```

---

# 5. Example

Imagine your knowledge base contains this document:

```text
Transformers were introduced in the paper
"Attention Is All You Need."

The architecture uses self-attention mechanisms.

Transformers do not require recurrence like RNNs.

The model uses positional encoding to represent
token positions.

The original Transformer contains an encoder and decoder.

Transformers are widely used in NLP and generative AI.
```

User asks:

> **"Why don't Transformers need recurrence?"**

A normal retriever might return the entire chunk.

Contextual compression tries to retain the relevant portion:

```text
"Transformers do not require recurrence like RNNs."
```

Potentially along with nearby context if needed.

---

# 6. LangChain Implementation

LangChain provides:

```python
ContextualCompressionRetriever
```

You generally need:

1. A **base retriever**
2. A **document compressor**

Conceptually:

```python
compression_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)
```

Then:

```python
docs = compression_retriever.invoke(
    "What is self-attention?"
)
```

---

# 7. Example with an LLM Compressor

One common approach is an LLM-based compressor.

```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)
```

Then:

```python
docs = compression_retriever.invoke(
    "What is self-attention?"
)
```

The flow becomes:

```text
Question
   ↓
Base Retriever
   ↓
Documents
   ↓
LLM Compressor
   ↓
Relevant Extracts
```

---

# 8. What Does `LLMChainExtractor` Do?

It asks the LLM something conceptually similar to:

> "Given this document and the user's question, extract only the portions of the document that are relevant to answering the question."

For example:

```text
Question:
What is self-attention?

Document:
[500 words about Transformers]

             ↓

Compressor

             ↓

Relevant content:
"Self-attention allows each token to
consider information from other tokens..."
```

The LLM is **not necessarily answering the user** at this stage.

It is primarily **extracting relevant context**.

---

# 9. Important: Retrieval and Compression Are Different

This distinction matters.

### Retriever

Answers:

> **Which documents might contain the answer?**

```text
Query
 ↓
Retriever
 ↓
Documents
```

### Compressor

Answers:

> **Which parts of those documents are actually useful for this query?**

```text
Query + Documents
 ↓
Compressor
 ↓
Relevant Content
```

### LLM

Answers:

> **What is the final answer?**

```text
Query + Relevant Context
 ↓
LLM
 ↓
Answer
```

---

# 10. Complete RAG Pipeline

With contextual compression:

```text
                    User Question
                          │
                          ▼
                    Base Retriever
                          │
                          ▼
                  Retrieved Documents
                          │
                          ▼
                Contextual Compressor
                          │
                          ▼
                  Relevant Information
                          │
                          ▼
                       Prompt
                          │
                          ▼
                         LLM
                          │
                          ▼
                       Answer
```

---

# 11. Why Not Just Set `k=2`?

You might think:

> "Why not retrieve fewer documents?"

Because **number of documents and amount of irrelevant content are different problems**.

Suppose:

```text
k = 2
```

and each document contains:

```text
5,000 words
```

You still have:

```text
10,000 words
```

of context.

Contextual compression can instead do:

```text
2 Documents
       ↓
Extract relevant sections
       ↓
500 words
```

So:

```text
k controls:
How many documents?

Compression controls:
How much useful content from those documents?
```

---

# 12. Types of Compressors

Contextual compression is an **architecture**, not one single compression algorithm.

Different compressors can be used.

Common approaches include:

```text
Contextual Compression
│
├── LLMChainExtractor
├── LLMChainFilter
├── EmbeddingsFilter
└── DocumentCompressorPipeline
```

---

# 13. LLMChainExtractor

This extracts relevant portions of documents.

```text
Document
   ↓
LLM
   ↓
Relevant passages
```

Example:

```text
Original:
1000 words

       ↓

Extract relevant information

       ↓

150 words
```

Good when you want **fine-grained extraction**.

---

# 14. LLMChainFilter

Instead of extracting pieces of a document, the LLM can decide whether the **whole document should be kept or discarded**.

```text
Document 1 → Relevant → KEEP
Document 2 → Irrelevant → REMOVE
Document 3 → Relevant → KEEP
Document 4 → Irrelevant → REMOVE
```

So:

### Extractor

```text
Keep part of a document
```

### Filter

```text
Keep or discard the entire document
```

---

# 15. EmbeddingsFilter

You can also use embeddings rather than an LLM to filter content.

Conceptually:

```text
Query
 ↓
Embedding
 ↓
Compare with document/chunk embeddings
 ↓
Remove low-relevance content
```

This can be cheaper than an LLM-based compressor.

For example:

```text
Query
 ↓
Embedding similarity
 ↓
Threshold = 0.75
 ↓
Keep sufficiently relevant chunks
```

---

# 16. Document Compressor Pipeline

You can combine multiple compression techniques.

For example:

```text
Retrieved Documents
        ↓
Embedding Filter
        ↓
Remove obviously irrelevant docs
        ↓
LLM Extractor
        ↓
Extract relevant passages
        ↓
LLM
```

Architecture:

```text
Retriever
   ↓
Compressor 1
   ↓
Compressor 2
   ↓
Compressor 3
   ↓
Final Context
```

This is useful in more advanced RAG systems.

---

# 17. Contextual Compression vs MMR

You just learned MMR, so don't confuse them.

### MMR

Works primarily during **document selection**.

```text
Query
 ↓
Candidate Documents
 ↓
MMR
 ↓
Relevant + Diverse Documents
```

Goal:

> **Reduce redundancy while maintaining relevance.**

---

### Contextual Compression

Works **after documents have been retrieved**.

```text
Query
 ↓
Retriever
 ↓
Documents
 ↓
Compression
 ↓
Relevant passages
```

Goal:

> **Remove irrelevant content from retrieved documents.**

---

## Comparison

|                      | MMR                       | Contextual Compression      |
| -------------------- | ------------------------- | --------------------------- |
| Main goal            | Diversity                 | Reduce irrelevant content   |
| Reduces redundancy   | Yes                       | Can                         |
| Operates on          | Candidate documents       | Retrieved documents/content |
| Uses LLM             | Not required              | Often                       |
| Reduces context size | Sometimes                 | Yes                         |
| Main benefit         | Better document selection | Better context quality      |

---

# 18. Contextual Compression vs Multi-Query

### Multi-Query

```text
One Question
      ↓
Multiple Questions
      ↓
Multiple Searches
      ↓
More Relevant Documents
```

Goal:

> **Increase recall.**

### Contextual Compression

```text
Question
      ↓
Retrieve Documents
      ↓
Compress Documents
      ↓
Relevant Content
```

Goal:

> **Improve precision and reduce context.**

They can be combined:

```text
User Query
    ↓
Multi-Query
    ↓
Multiple Retrievals
    ↓
MMR / Ranking
    ↓
Contextual Compression
    ↓
LLM
```

---

# 19. When Should You Use Contextual Compression?

It is especially useful when your documents are:

### Large

```text
Long PDFs
Books
Research papers
Documentation
```

### Information-dense

A single chunk may contain multiple unrelated topics.

### Retrieved chunks contain lots of irrelevant text

For example:

```text
Chunk size = 2,000 tokens

Relevant information = 200 tokens
```

Compression can significantly improve the context sent to the LLM.

---

# 20. Advantages

### 1. Smaller context

```text
Less irrelevant text
```

### 2. Lower token usage

Potentially reduces LLM input-token consumption.

### 3. Better focus

The generation model sees information more directly related to the question.

### 4. Works with existing retrievers

You can put it on top of:

```text
Chroma
FAISS
BM25
Wikipedia
etc.
```

Conceptually:

```text
Any Retriever
      ↓
Contextual Compression
      ↓
LLM
```

---

# 21. Disadvantages

### 1. Additional latency

If an LLM performs compression:

```text
Retriever
   ↓
LLM Compressor
   ↓
Generation LLM
```

you have an additional model call.

### 2. Additional cost

The compressor itself may consume tokens.

### 3. Information can be accidentally removed

An overly aggressive compressor could remove information that turns out to be important.

### 4. More complexity

Your RAG pipeline becomes more sophisticated.

---

# 22. Simple Mental Model

Think about a library.

You ask:

> "What is the definition of RAG?"

The Retriever gives you:

```text
📚 Book
Page 100
Page 101
Page 102
Page 103
```

Contextual Compression says:

> "I don't need all four pages. Give me only the paragraphs relevant to the definition of RAG."

So:

```text
Library
  ↓
Retriever
  ↓
Relevant Pages
  ↓
Compressor
  ↓
Relevant Paragraphs
  ↓
LLM
```

---

# 23. Interview Answer

If asked:

> **What is a Contextual Compression Retriever?**

A strong answer is:

> **A Contextual Compression Retriever is a LangChain retrieval approach that combines a base retriever with a document compressor. The base retriever first retrieves potentially relevant documents, and the compressor then filters or extracts only the portions relevant to the user's query. This reduces irrelevant context, improves the quality of information passed to the LLM, and can reduce token usage.**

---

# Final Mental Model

You should now distinguish the three retrieval strategies:

```text
                    RAG Retrieval
                         │
       ┌─────────────────┼─────────────────┐
       ▼                 ▼                 ▼
   Multi-Query          MMR        Contextual Compression
       │                 │                 │
       ▼                 ▼                 ▼
 Increase Recall     Diversity       Remove Irrelevant
       │                 │                 │
       ▼                 ▼                 ▼
 Multiple Queries   Less Redundancy   Smaller Context
```

The complete advanced pipeline can be:

```text
User Question
     │
     ▼
Multi-Query
     │
     ▼
Multiple Searches
     │
     ▼
MMR / Ranking
     │
     ▼
Candidate Documents
     │
     ▼
Contextual Compression
     │
     ▼
Relevant Passages
     │
     ▼
LLM
     │
     ▼
Final Answer
```

**Core takeaway:**

> **Retriever finds relevant documents; MMR selects relevant and diverse documents; Multi-Query searches from multiple perspectives; Contextual Compression removes irrelevant content from the retrieved documents before the LLM sees it.**
